# Exercise 1. Prompting!
LLMs generate words (tokens) probabilistically in various ways, depending on the architecture and sampling method. As a consequence, the we words choose to provide the LLM impact what it generates. 

The process of "choosing the right words"  is called **prompt engineering**

:::{admonition} PAPER SPOTLIGHT
:class: fucsia, dropdown
If you're interested in this topic, I suggest reading:
> [What’s the Difference? Supporting Users in Identifying the Effects of
Prompt and Model Changes Through Token Patterns](https://aclanthology.org/2025.acl-long.985.pdf)" by {cite:t}`hedderich-etal-2025-whats`
:::

## 1.1 Setup: Import Packages
If you have not already, please download the packages below (in venv or in UCloud) in your terminal:

```bash
pip install transformers torch
```

:::{admonition} Or download in notebook ... 
:class: tip, dropddown Remember, you can also download the packages in Jupyter notebooks with the %pip magic command as we have done in previous classes. 
:::

Import the packages

In [14]:
from transformers import AutoTokenizer
import transformers 
import torch 

## 1.2 Load Models
Let's load Google's `flan-t5-base` and OpenAI's `gpt2`. These models are a bit older, which means that we'll need to be extra clever to get them to do what we want :) 

We'll use `transformers.pipeline` to load the models:

In [15]:
max_length = 250 # how many tokens max to generate

When loading `FLAN-T5`, we specify the task `text2text-generation`:

In [16]:
model = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model)
pipeline_t5 = transformers.pipeline(
    task = "text2text-generation",
    model=model,
    dtype=torch.float16, # a way to reduce memory
    max_length=max_length,
)

Device set to use mps:0


For `GPT-2`, the task is `text-generation`. The difference has something to do with the type of transformer architecture that makes up the models.

In [19]:
model = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model)
pipeline_gpt = transformers.pipeline(
    task = "text-generation",
    model=model,
    dtype=torch.float16,
    max_length=max_length
)

Device set to use mps:0


:::{admonition} QUESTION
:class: red
Which component(s) make up the architecture of these models? Decoder-only? Encoder-decoder? Encoder-only? 

Discuss with a friend and google it if you don't know, then check the answers below.

<details>
<summary>ANSWER</summary>
GPT2 is decoder-only! (<a href="https://jalammar.github.io/illustrated-gpt2/">See here</a>)     

FLAN-T5 is encoder-decoder (<a href = "https://huggingface.co/docs/transformers/en/model_doc/t5">See here</a>)
</details>
:::

## 1.3 Text Completion
Let's try to ask Flan-T5 a simple question:

In [48]:
pipeline_t5("What is the capital of Denmark?")

[{'generated_text': 'djurgrden'}]

The generated text above was obviously not what we wanted, let's phrase it in another way:

In [43]:
pipeline_t5("The capital of Denmark is")

[{'generated_text': 'Copenhagen'}]

:::{admonition} QUESTION
:class: red
Do you have any idea why the first phrasing, but the second one worked?

<details>
<summary>ANSWER</summary>
Flan-T5, like other language models, predicts the next word based on previous words.        
<br><br>
The second phrasing, "The capital of Denmark is", is an incomplete sentence, which naturally prompts the model to predict "Copenhagen" as the next word. The first phrasing is a complete sentence, so it is less likely that the next word is "Copenhagen" in a natural text sequence.
</details>
:::

### 1.4 Summarization
A very useful application of LLMs is summarization! As `Flan-T5` is instruction-tuned...